# 02 — Chunking & Indexing (Layout X + Layout Y)

**Audience:** Engineers / SEs following the technical walkthrough.

**What this notebook shows:**

1. Sentence-aware **chunking** of parsed documents (`data/parsed/<doc>/layout.json` → `data/chunks/<doc>.jsonl`).
2. **Dual-layout indexing** into Azure AI Search:
   - **Layout X** — `tax-chunks` index: chunk vectors with nested sentences (primary retrieval surface).
   - **Layout Y** — `tax-sentences` index: sentence-level docs materialized via index projections (sentence-primary re-ranking / citation surface).
3. **Round-trip verification** — pull a known `chunk_id` / `sentence_id` from each index to confirm both are live.

**Prerequisites:**

- You ran `01_ingest_and_parse.ipynb` so `data/parsed/<doc>/layout.json` exists, **OR**
- Set `RESUME_FROM_CACHE = True` below to use the cached bundle under `data/notebook_cache/` (`parsed/` + `chunks/`). The cache bundle is being produced in parallel; assume it exists at `data/notebook_cache/chunks/` and `data/notebook_cache/parsed/`.
- `.env` is populated; `DefaultAzureCredential` can reach the Search + embeddings endpoints.

Design rationale for the two indexes lives in [`docs/index_projections_eval.md`](../docs/index_projections_eval.md).

In [ ]:
# --- Top-level knobs --------------------------------------------------------
# Flip to False to force re-chunking from data/parsed/ instead of using the
# pre-built cache at data/notebook_cache/.
RESUME_FROM_CACHE = True

# Cost/scale knob. Full corpus = 2,558 chunks + ~64k sentences, which will take
# real minutes + real $ on the embeddings endpoint. Default to a 2-doc sample.
# Set to None to index every document we have chunks for.
LIMIT_DOCS = 2   # e.g. set to None once you're ready to scale up

# If the indexes are already populated (Azure reality-check cell below), we
# skip the expensive re-upload by default.
FORCE_REUPLOAD = False

## Part 1 — Chunking

Turns the `layout.json` from Stage 1 into `data/chunks/<doc_id>.jsonl`.
Each line is a `Chunk` with its `sentences` array; `sentence_id`s are
doc-scoped and stable, and `text[char_start:char_end]` round-trips to
the sentence text on every record.

Chunking policy (set in `src/sentcite/chunking.py`):

- spaCy `sentencizer` (no model download).
- `tiktoken` `cl100k_base` for token counting (GPT-4o/4.1 family).
- Target ~400 tokens, ~40 token (≈10 %) overlap at sentence boundaries.
- Chunks never cross a section heading — keeps `section_path` faithful.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from sentcite.chunking import chunk_document
from sentcite.indexing import load_chunks_from_jsonl

PARSED_DIR = REPO_ROOT / ('data/notebook_cache/parsed' if RESUME_FROM_CACHE else 'data/parsed')
CHUNKS_DIR = REPO_ROOT / ('data/notebook_cache/chunks' if RESUME_FROM_CACHE else 'data/chunks')
print(f'PARSED_DIR = {PARSED_DIR}  (exists={PARSED_DIR.exists()})')
print(f'CHUNKS_DIR = {CHUNKS_DIR}  (exists={CHUNKS_DIR.exists()})')

DOC_ID = 'irs_pub_583'
layout = json.loads((PARSED_DIR / DOC_ID / 'layout.json').read_text())
chunks = chunk_document(DOC_ID, layout)
{
    'doc': DOC_ID,
    'chunks': len(chunks),
    'sentences': sum(len(c.sentences) for c in chunks),
    'mean_tokens_per_chunk': round(sum(c.token_count for c in chunks) / max(len(chunks), 1), 1),
    'distinct_sections': len({tuple(c.section_path) for c in chunks}),
}

In [ ]:
# Verify sentence offsets round-trip inside each chunk.
bad = [
    (c.chunk_id, s.sentence_id)
    for c in chunks
    for s in c.sentences
    if c.text[s.char_start:s.char_end] != s.text
]
f'round-trip failures: {len(bad)}'

In [ ]:
# Peek at a mid-document chunk and its first few sentences.
c = chunks[len(chunks) // 3]
{
    'chunk_id': c.chunk_id,
    'page': c.page,
    'token_count': c.token_count,
    'section_path': c.section_path,
    'n_sentences': len(c.sentences),
    'first_3': [s.text for s in c.sentences[:3]],
}

In [ ]:
# Confirm sentence overlap across adjacent chunks within the same section:
# the last sentence_id of chunk N should reappear as the first sentence_id
# of chunk N+1 whenever they share a section_path.
examples = []
for a, b in zip(chunks, chunks[1:]):
    if tuple(a.section_path) != tuple(b.section_path):
        continue
    ids_a = {s.sentence_id for s in a.sentences[-5:]}
    ids_b = {s.sentence_id for s in b.sentences[:5]}
    overlap = ids_a & ids_b
    if overlap:
        examples.append((a.chunk_id, b.chunk_id, sorted(overlap)))
    if len(examples) >= 3:
        break
examples

## Part 2 — Indexing (Layout X + Layout Y)

We maintain **two** Azure AI Search indexes side by side. They share the same embedding model and upstream chunk JSONL, but project the data at different granularities:

| Index | Layout | Primary doc | Vector field | Use |
| --- | --- | --- | --- | --- |
| `tax-chunks`    | **X** | chunk    | chunk embedding    | main retrieval; nested `sentences` field carries per-sentence text + offsets for in-place citation. |
| `tax-sentences` | **Y** | sentence | sentence embedding | sentence-level re-ranking / fine-grained citation alignment via index projections. |

Full write-up (why both, not just one) is in [`docs/index_projections_eval.md`](../docs/index_projections_eval.md).

> ⚠️ **Downstream notebooks (03 query+generate, 04 citation alignment, 05 eval) assume BOTH indexes are built.** `sentcite.retrieval.retrieve()` defaults to `mode="dual"` and will raise if `tax-sentences` is missing. **Don't skip Layout Y.**

In [ ]:
# Azure reality check: are the indexes already populated? If so, we skip the
# expensive upload and just use what's there (set FORCE_REUPLOAD=True above
# to override).
from sentcite.config import AzureConfig
from sentcite.indexing import (
    ensure_chunks_index,
    ensure_sentences_index,
    upload_chunks,
    upload_sentences,
    _search_client,
    _sentences_client,
)

cfg = AzureConfig.from_env()

def _doc_count(client):
    try:
        return client.get_document_count()
    except Exception as exc:  # index may not exist yet
        return f'(unavailable: {type(exc).__name__})'

chunks_client = _search_client(cfg)
sentences_client = _sentences_client(cfg)
pre = {
    'tax-chunks doc_count': _doc_count(chunks_client),
    'tax-sentences doc_count': _doc_count(sentences_client),
}
pre

In [ ]:
# Ensure both index schemas exist (idempotent; does NOT drop data).
x_name = ensure_chunks_index(cfg)
y_name = ensure_sentences_index(cfg)
print(f'Layout X index ready: {x_name!r}')
print(f'Layout Y index ready: {y_name!r}')

In [ ]:
# Select the JSONL files to upload. LIMIT_DOCS keeps the sample cheap.
jsonl_files = sorted(CHUNKS_DIR.glob('*.jsonl'))
if LIMIT_DOCS is not None:
    jsonl_files = jsonl_files[:LIMIT_DOCS]
print(f'{len(jsonl_files)} JSONL files will be indexed:')
for p in jsonl_files:
    print(f'  - {p.name}')

In [ ]:
# Upload into BOTH indexes. Skips automatically if pre-existing doc counts look
# healthy, unless FORCE_REUPLOAD is set.
import time

existing_chunks = pre['tax-chunks doc_count'] if isinstance(pre['tax-chunks doc_count'], int) else 0
existing_sents = pre['tax-sentences doc_count'] if isinstance(pre['tax-sentences doc_count'], int) else 0

if not FORCE_REUPLOAD and existing_chunks > 0 and existing_sents > 0:
    print(f'[skip] tax-chunks={existing_chunks}, tax-sentences={existing_sents} — set FORCE_REUPLOAD=True to re-upload.')
else:
    totals = {'chunks': 0, 'sent_nested': 0, 'sent_index': 0}
    for path in jsonl_files:
        doc_chunks = load_chunks_from_jsonl(path)
        t0 = time.perf_counter()
        x_counts = upload_chunks(doc_chunks, cfg=cfg)          # Layout X
        y_counts = upload_sentences(doc_chunks, cfg=cfg)        # Layout Y
        dt = time.perf_counter() - t0
        totals['chunks']      += x_counts['chunks']
        totals['sent_nested'] += x_counts['sentences']
        totals['sent_index']  += y_counts['sentences']
        print(
            f'[upload] {path.stem:16s} chunks={x_counts["chunks"]:4d} '
            f'sent_nested={x_counts["sentences"]:5d} '
            f'sent_index={y_counts["sentences"]:5d} elapsed={dt:6.1f}s'
        )
    print(f'[done] {totals}')

### Round-trip verification

Grab a known `chunk_id` + `sentence_id` from the local JSONL, then fetch them back out of each Azure index. If both lookups return the record, the pipeline is genuinely live — not just "the API call didn't throw".

In [ ]:
# Pick a known chunk + sentence from the first JSONL we indexed.
sample_doc = load_chunks_from_jsonl(jsonl_files[0])
sample_chunk = sample_doc[len(sample_doc) // 2]
sample_sentence = sample_chunk.sentences[0]
print(f'probe chunk_id    = {sample_chunk.chunk_id}')
print(f'probe sentence_id = {sample_sentence.sentence_id}')

In [ ]:
# Layout X round-trip: fetch the chunk doc by key.
x_hit = chunks_client.get_document(key=sample_chunk.chunk_id)
{
    'chunk_id': x_hit.get('chunk_id'),
    'document_id': x_hit.get('document_id'),
    'section_path': x_hit.get('section_path'),
    'token_count': x_hit.get('token_count'),
    'nested_sentences': len(x_hit.get('sentences') or []),
    'text_preview': (x_hit.get('text') or '')[:160] + '…',
}

In [ ]:
# Layout Y round-trip: fetch the sentence doc by key.
y_hit = sentences_client.get_document(key=sample_sentence.sentence_id)
{
    'sentence_id': y_hit.get('sentence_id'),
    'parent_chunk_id': y_hit.get('parent_chunk_id') or y_hit.get('chunk_id'),
    'document_id': y_hit.get('document_id'),
    'section_path': y_hit.get('section_path'),
    'text': y_hit.get('text'),
}

## What's next

With both indexes live, continue to [`03_query_and_generate.ipynb`](03_query_and_generate.ipynb) — that notebook drives `sentcite.retrieval.retrieve(mode="dual")` against Layout X + Layout Y together and hands the hits to the grounded generator.

If you stopped at `LIMIT_DOCS = 2`, bump it (or set to `None`) and re-run the upload cell before moving on if you want end-to-end numbers on the full corpus.